In [1]:
%pip -q install duckdb pyarrow

from google.colab import drive
from pathlib import Path
import os
import shutil

import duckdb
import pandas as pd
import pyarrow.parquet as pq

try:
    drive.flush_and_unmount()
except Exception:
    pass

if os.path.exists("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)

drive.mount(
    "/content/drive",
    force_remount=True,
    timeout_ms=300000,
)

DATA_DIR = Path("/content/drive/MyDrive/Language Detection")
PARQUET_PATH = DATA_DIR / "sessions_lang_transcript_2026-08-23_2026-08-24.parquet"

if not DATA_DIR.exists():
    raise FileNotFoundError(DATA_DIR)

if not PARQUET_PATH.is_file():
    raise FileNotFoundError(PARQUET_PATH)

parquet_file = pq.ParquetFile(PARQUET_PATH)
metadata = parquet_file.metadata

print("File path  :", PARQUET_PATH)
print("File size  :", f"{PARQUET_PATH.stat().st_size / (1024**2):.2f} MB")
print("Rows       :", f"{metadata.num_rows:,}")
print("Row groups :", metadata.num_row_groups)
print("Columns    :", metadata.num_columns)


Mounted at /content/drive
File path  : /content/drive/MyDrive/Language Detection/sessions_lang_transcript_2026-08-23_2026-08-24.parquet
File size  : 469.49 MB
Rows       : 3,469
Row groups : 1
Columns    : 12


In [2]:
LANGUAGE_CODES = ("en", "de", "fr", "pt", "es", "ru")
SESSIONS_PER_LANGUAGE = 5
SEGMENTS_PER_SESSION = 5
SEGMENTS_PER_LANGUAGE = SESSIONS_PER_LANGUAGE * SEGMENTS_PER_SESSION
MIN_WORDS = 4
MAX_WORDS = 12
MIN_SESSION_SEGMENTS = 25
MIDDLE_START = 0.25
MIDDLE_END = 0.75

DB_PATH = Path("/content/language_review.duckdb")
TEMP_DIR = Path("/content/duckdb_tmp")

TEMP_DIR.mkdir(parents=True, exist_ok=True)

if DB_PATH.exists():
    DB_PATH.unlink()

con = duckdb.connect(DB_PATH.as_posix())
con.execute("SET threads TO 4")
con.execute("SET memory_limit = '4GB'")
con.execute(f"SET temp_directory = '{TEMP_DIR.as_posix()}'")
con.execute("SET preserve_insertion_order = false")

language_sql = ", ".join(f"'{code}'" for code in LANGUAGE_CODES)

con.execute(
    f"""
    CREATE TABLE segment_pool AS
    WITH source AS (
        SELECT
            gamesession_id,
            url,
            TRY_CAST(created_at AS TIMESTAMP) AS created_at,
            lang_detected,
            len(transcript_segments) AS session_segment_count,
            transcript_segments
        FROM read_parquet('{PARQUET_PATH.as_posix()}')
        WHERE
            lang_detected IN ({language_sql})
            AND transcript_segments IS NOT NULL
            AND len(transcript_segments) >= {MIN_SESSION_SEGMENTS}
    ),
    exploded AS (
        SELECT
            gamesession_id,
            url,
            created_at,
            lang_detected,
            session_segment_count,
            generate_subscripts(transcript_segments, 1) AS segment_index,
            UNNEST(transcript_segments) AS segment
        FROM source
    ),
    parsed AS (
        SELECT
            gamesession_id,
            url,
            created_at,
            lang_detected,
            session_segment_count,
            segment_index,
            TRY_CAST(segment.timestamp[1] AS DOUBLE) AS segment_start,
            TRY_CAST(segment.timestamp[2] AS DOUBLE) AS segment_end,
            TRIM(segment.text) AS segment_text,
            regexp_extract_all(
                TRIM(segment.text),
                '[\\p{{L}}\\p{{N}}][\\p{{L}}\\p{{M}}\\p{{N}}''’_-]*[\\p{{L}}\\p{{M}}\\p{{N}}]'
            ) AS valid_words
        FROM exploded
        WHERE
            segment.text IS NOT NULL
            AND TRIM(segment.text) <> ''
    ),
    filtered AS (
        SELECT
            gamesession_id,
            url,
            created_at,
            lang_detected,
            session_segment_count,
            segment_index,
            segment_start,
            segment_end,
            segment_text,
            len(valid_words) AS word_count,
            TRIM(
                regexp_replace(
                    lower(
                        regexp_replace(
                            segment_text,
                            '[^\\p{{L}}\\p{{M}}\\p{{N}}''’_-]+',
                            ' ',
                            'g'
                        )
                    ),
                    '\\s+',
                    ' ',
                    'g'
                )
            ) AS normalized_text,
            abs(
                segment_index
                - ((session_segment_count + 1) / 2.0)
            ) AS center_distance
        FROM parsed
        WHERE
            len(valid_words) BETWEEN {MIN_WORDS} AND {MAX_WORDS}
    ),
    bounded AS (
        SELECT *
        FROM filtered
        WHERE
            segment_start IS NOT NULL
            AND segment_end IS NOT NULL
            AND normalized_text <> ''
            AND segment_index >= ceil(session_segment_count * {MIDDLE_START})
            AND segment_index <= floor(session_segment_count * {MIDDLE_END})
    ),
    deduplicated AS (
        SELECT *
        FROM bounded
        QUALIFY row_number() OVER (
            PARTITION BY
                lang_detected,
                gamesession_id,
                normalized_text
            ORDER BY
                center_distance ASC,
                segment_index ASC
        ) = 1
    )
    SELECT *
    FROM deduplicated
    """
)

con.execute(
    """
    CREATE INDEX segment_pool_language_session_idx
    ON segment_pool(lang_detected, gamesession_id)
    """
)

def format_timestamp(seconds):
    total_seconds = int(float(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

def get_language_segments(language_code):
    if language_code not in LANGUAGE_CODES:
        raise ValueError(language_code)

    candidates = con.execute(
        """
        SELECT
            gamesession_id,
            url,
            created_at,
            segment_index,
            segment_start,
            segment_end,
            segment_text,
            normalized_text,
            word_count,
            lang_detected AS language_detected,
            center_distance
        FROM segment_pool
        WHERE lang_detected = ?
        ORDER BY
            created_at DESC,
            gamesession_id DESC,
            center_distance ASC,
            segment_index ASC
        """,
        [language_code],
    ).df()

    if candidates.empty:
        raise ValueError(f"No qualifying segments for {language_code}")

    session_order = (
        candidates[
            ["gamesession_id", "created_at"]
        ]
        .drop_duplicates("gamesession_id")
        .sort_values(
            ["created_at", "gamesession_id"],
            ascending=[False, False],
            na_position="last",
        )
        ["gamesession_id"]
        .tolist()
    )

    selected = []
    seen_transcripts = set()

    for gamesession_id in session_order:
        session_candidates = candidates[
            candidates["gamesession_id"] == gamesession_id
        ]

        session_candidates = session_candidates[
            ~session_candidates["normalized_text"].isin(seen_transcripts)
        ]

        if len(session_candidates) < SEGMENTS_PER_SESSION:
            continue

        picked = (
            session_candidates
            .head(SEGMENTS_PER_SESSION)
            .sort_values("segment_index")
            .reset_index(drop=True)
        )

        selected.append(picked)
        seen_transcripts.update(picked["normalized_text"])

        if len(selected) == SESSIONS_PER_LANGUAGE:
            break

    if len(selected) != SESSIONS_PER_LANGUAGE:
        raise ValueError(
            f"Not enough qualifying sessions for {language_code}: "
            f"{len(selected)}/{SESSIONS_PER_LANGUAGE}"
        )

    result = pd.concat(selected, ignore_index=True)

    if len(result) != SEGMENTS_PER_LANGUAGE:
        raise ValueError(
            f"Expected {SEGMENTS_PER_LANGUAGE} segments for "
            f"{language_code}, found {len(result)}"
        )

    session_counts = result.groupby("gamesession_id").size()

    if len(session_counts) != SESSIONS_PER_LANGUAGE:
        raise ValueError(
            f"Expected {SESSIONS_PER_LANGUAGE} sessions for {language_code}"
        )

    if not (session_counts == SEGMENTS_PER_SESSION).all():
        raise ValueError(
            f"Each session must contribute {SEGMENTS_PER_SESSION} segments"
        )

    if result["normalized_text"].duplicated().any():
        raise ValueError(
            f"Duplicate transcript detected for {language_code}"
        )

    result["segment_timestamp"] = (
        result["segment_start"].map(format_timestamp)
        + " - "
        + result["segment_end"].map(format_timestamp)
    )

    result["verdict"] = pd.Series(
        "",
        index=result.index,
        dtype="string",
    )

    result["note"] = pd.Series(
        "",
        index=result.index,
        dtype="string",
    )

    return result[
        [
            "gamesession_id",
            "url",
            "segment_index",
            "segment_timestamp",
            "segment_text",
            "language_detected",
            "verdict",
            "note",
        ]
    ]

def show_segments(frame):
    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
        "display.max_colwidth", None,
        "display.width", None,
    ):
        display(frame)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
english_segments = get_language_segments("en")
show_segments(english_segments)


,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,verdict,note
0,141268501,https://www.twitch.tv/videos/2854311727,225,00:38:11 - 00:38:16,I take the plane. I can knock a plane.,en,,
1,141268501,https://www.twitch.tv/videos/2854311727,228,00:38:33 - 00:38:37,Your attempt on your eye. Your eyes is yellow.,en,,
2,141268501,https://www.twitch.tv/videos/2854311727,230,00:39:00 - 00:39:04,I've got to be able to make this cartoon bigger.,en,,
3,141268501,https://www.twitch.tv/videos/2854311727,231,00:39:05 - 00:39:09,"Oh, there's spots high at see on that heavy.",en,,
4,141268501,https://www.twitch.tv/videos/2854311727,233,00:39:20 - 00:39:23,You've here for a payment. I just need get to the safe zone.,en,,
5,141268683,https://www.twitch.tv/videos/2854243785,197,01:54:55 - 01:54:57,I found a red access card.,en,,
6,141268683,https://www.twitch.tv/videos/2854243785,198,01:55:05 - 01:55:09,Did I blow your head off? bad.,en,,
7,141268683,https://www.twitch.tv/videos/2854243785,200,01:55:20 - 01:55:24,Heh. He to your Twix. following your Twix.,en,,
8,141268683,https://www.twitch.tv/videos/2854243785,201,01:55:26 - 01:55:29,"All your friends are going be like, oh, I'm go on one.",en,,
9,141268683,https://www.twitch.tv/videos/2854243785,202,01:55:42 - 01:55:50,Really Scull me. Scull follows me.,en,,


In [4]:
german_segments = get_language_segments("de")
show_segments(german_segments)


,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,verdict,note
0,141192575,https://www.twitch.tv/videos/2852368077,171,00:41:48 - 00:41:52,"Ich bleib jetzt da drüben, okay. Das hängt eh so.",de,,
1,141192575,https://www.twitch.tv/videos/2852368077,172,00:41:54 - 00:41:58,"Ich glaub, die ist safe. Boah, wo denn?",de,,
2,141192575,https://www.twitch.tv/videos/2852368077,175,00:42:14 - 00:42:19,Für mich auch. Weißt du? Da nicht gerade einer.,de,,
3,141192575,https://www.twitch.tv/videos/2852368077,176,00:42:24 - 00:42:29,"Ja, ich kann nix da. ist nicht so schön.",de,,
4,141192575,https://www.twitch.tv/videos/2852368077,177,00:42:29 - 00:42:33,Bleib weg. Was hat's hier? Wir haben da eins oben.,de,,
5,141268194,https://www.twitch.tv/videos/2854141333,660,02:37:08 - 02:37:09,Was ist Schiff denn?,de,,
6,141268194,https://www.twitch.tv/videos/2854141333,662,02:38:47 - 02:38:52,"ist denn mit dem Loch los? Achso, das ist ein Stein.",de,,
7,141268194,https://www.twitch.tv/videos/2854141333,664,02:39:21 - 02:39:24,Mhm. Vor allem Kapazität.,de,,
8,141268194,https://www.twitch.tv/videos/2854141333,666,02:40:14 - 02:40:18,"Aber jetzt ist aber mein Rotpunktvisier auch weg, Haha.",de,,
9,141268194,https://www.twitch.tv/videos/2854141333,667,02:40:18 - 02:40:21,"Ja. Danke, Spiel, ey.",de,,


In [5]:
french_segments = get_language_segments("fr")
show_segments(french_segments)


,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,verdict,note
0,141267561,https://www.twitch.tv/videos/2854116559,698,03:26:31 - 03:26:35,putain mais mec ! T'as rencontré une Roberto !,fr,,
1,141267561,https://www.twitch.tv/videos/2854116559,699,03:26:35 - 03:26:39,Terrible ! Oh putain mais t réussi à'en...,fr,,
2,141267561,https://www.twitch.tv/videos/2854116559,700,03:26:39 - 03:26:46,à t'en défaire ou pas ? C'est toujours pas fini !,fr,,
3,141267561,https://www.twitch.tv/videos/2854116559,701,03:26:46 - 03:26:51,Oh putain qu'est qu va nous dire ? Oui j'ai réussi !,fr,,
4,141267561,https://www.twitch.tv/videos/2854116559,702,03:26:51 - 03:26:56,Ok ! Mais qu'est-ce qui n pas fini ? Ça m'inquiète !,fr,,
5,141268545,https://www.twitch.tv/videos/2854355409,31,00:10:03 - 00:10:12,putain quel campeur il est pas là,fr,,
6,141268545,https://www.twitch.tv/videos/2854355409,38,00:12:24 - 00:12:26,Ça pop à côté Faut,fr,,
7,141268545,https://www.twitch.tv/videos/2854355409,39,00:13:05 - 00:13:08,"t'arrêter,'es Faut t'es sous,",fr,,
8,141268545,https://www.twitch.tv/videos/2854355409,40,00:13:16 - 00:13:20,Nice Il en un sur la tour,fr,,
9,141268545,https://www.twitch.tv/videos/2854355409,41,00:13:29 - 00:13:34,Ça arrive c'est nous En bas,fr,,


In [6]:
portuguese_segments = get_language_segments("pt")
show_segments(portuguese_segments)


,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,verdict,note
0,141268410,https://www.twitch.tv/videos/2854267348,252,01:18:27 - 01:18:32,Obrigado Worma por o raid. Obrigado muito. Quem faz um shout-out a Worma?,pt,,
1,141268410,https://www.twitch.tv/videos/2854267348,253,01:18:32 - 01:18:36,que não tem ninguém... Ninguém... Como se chama?,pt,,
2,141268410,https://www.twitch.tv/videos/2854267348,254,01:18:36 - 01:18:39,Ninguém está agora. Eu que fazer mas... Eu que primeiro aclarar-se.,pt,,
3,141268410,https://www.twitch.tv/videos/2854267348,255,01:18:39 - 01:18:43,"Me desculpe, Worma. Bem-vindos a todos os Wormini, ou...",pt,,
4,141268410,https://www.twitch.tv/videos/2854267348,256,01:18:43 - 01:18:47,Como se chama? se chama? Todos os Vermant.,pt,,
5,141266587,https://www.twitch.tv/videos/2854208164,464,01:06:33 - 01:06:37,"Certo, ele caiu. Calma, volta, calma.",pt,,
6,141266587,https://www.twitch.tv/videos/2854208164,465,01:06:38 - 01:06:43,"Ah, bomba. Ele tá indo, ele tá indo.",pt,,
7,141266587,https://www.twitch.tv/videos/2854208164,466,01:06:44 - 01:06:47,"Tem spot. Spot, spot.",pt,,
8,141266587,https://www.twitch.tv/videos/2854208164,473,01:07:43 - 01:07:48,Eu tô vendo quem é ruim de nós dois aqui. Aperta a tab.,pt,,
9,141266587,https://www.twitch.tv/videos/2854208164,474,01:07:48 - 01:07:52,"Pensa a porta aí. Bora, Neve, chega aí.",pt,,


In [7]:
spanish_segments = get_language_segments("es")
show_segments(spanish_segments)


,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,verdict,note
0,141268448,https://www.twitch.tv/videos/2853628924,470,01:29:50 - 01:29:53,¿Por qué no vas ver si alguien necesita ayuda? ¿Ayuda?,es,,
1,141268448,https://www.twitch.tv/videos/2853628924,471,01:29:55 - 01:29:59,"Nueva misión disponible. la ayuntamiento. Vale, ver, ver, a ver.",es,,
2,141268448,https://www.twitch.tv/videos/2853628924,472,01:30:00 - 01:30:04,Tengo que escucharme cantar Tengo que escucharme cantar,es,,
3,141268448,https://www.twitch.tv/videos/2853628924,479,01:31:35 - 01:31:38,"Además, busquen a los sifantes y monten las sierras altas, están mejor que nunca.",es,,
4,141268448,https://www.twitch.tv/videos/2853628924,480,01:31:38 - 01:31:44,"Un momento. ¿Dónde está Don? Puntual. Garfas, Archizeli.",es,,
5,141267802,https://www.twitch.tv/videos/2854317733,59,00:14:55 - 00:15:04,estamos atrapados Smith agarra disparos al muy rápido para su oponente ahora,es,,
6,141267802,https://www.twitch.tv/videos/2854317733,60,00:15:21 - 00:15:28,no puede los dos lados,es,,
7,141267802,https://www.twitch.tv/videos/2854317733,63,00:16:20 - 00:16:28,Izquierda Y te vas sometido No te tires al suelo contra mí,es,,
8,141267802,https://www.twitch.tv/videos/2854317733,65,00:16:39 - 00:16:43,Vaya sumisión Vaya sumisión la acabamos de meter.,es,,
9,141267802,https://www.twitch.tv/videos/2854317733,66,00:16:47 - 00:16:52,"Sí, señor. Los tiré del suelo contra mí, que soy un grappler.",es,,


In [8]:
russian_segments = get_language_segments("ru")
show_segments(russian_segments)


,gamesession_id,url,segment_index,segment_timestamp,segment_text,language_detected,verdict,note
0,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,675,01:54:58 - 01:55:02,Куда? устал. есть батарейка если что?,ru,,
1,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,676,01:55:02 - 01:55:05,"Слушайте, держи телефон. У меня с собой банка.",ru,,
2,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,677,01:55:06 - 01:55:10,А как мы приклеим на эту камеру? Есть банка с этим.,ru,,
3,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,678,01:55:11 - 01:55:16,"Ебать ты функциональный человек. Есть такая банка, второй телефон.",ru,,
4,141267623,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454,680,01:55:38 - 01:55:47,"Ебать ты Илон Маскс Так, напиши Блять Мы вышли в курилку, я что-то",ru,,
5,141266950,https://www.twitch.tv/videos/2854259378,102,00:27:54 - 00:27:59,"Пусть поубивают там кто-то сгонит, чтобы нас упрать. Откуда ты?",ru,,
6,141266950,https://www.twitch.tv/videos/2854259378,103,00:28:00 - 00:28:03,"Сейчас уйдите там. А, окей, окей.",ru,,
7,141266950,https://www.twitch.tv/videos/2854259378,104,00:28:06 - 00:28:09,"Русы ж, пром. Русы ж, хуюсишь, блядь?",ru,,
8,141266950,https://www.twitch.tv/videos/2854259378,105,00:28:09 - 00:28:13,"Ну, я, я, лайфстрим, меня лайфстрим, они пишут там по-немецки.",ru,,
9,141266950,https://www.twitch.tv/videos/2854259378,106,00:28:14 - 00:28:17,"А, все, я понял, я так градусе не буду. Да, не, все нормально.",ru,,
